# 07. GraphSLAM 기초

GraphSLAM은 pose와 landmark를 노드로 두고, odometry/measurement constraint를 edge로 둔 뒤 전체 오차를 최소화한다.

$$x^* = \arg\min_x \sum_i e_i(x)^T\Omega_i e_i(x)$$

여기서는 1D pose graph로 정보행렬과 least squares 구조를 확인한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 1D Pose Graph Least Squares

연속 pose 사이 odometry constraint와 loop closure constraint를 함께 풀어 누적 drift를 보정한다.

In [ ]:
np.random.seed(11)
N=8
true=np.arange(N)*1.0
odom=np.diff(true)+np.random.randn(N-1)*0.08+0.04  # drift
edges=[]
for i,u in enumerate(odom):
    edges.append((i,i+1,u,1/0.08**2,'odom'))
edges.append((0,N-1,true[-1]-true[0]+np.random.randn()*0.05,1/0.05**2,'loop'))

# fix x0=0, solve x1..xN-1
A=[]; b=[]; labels=[]
for i,j,z,omega,label in edges:
    row=np.zeros(N-1)
    if i>0: row[i-1]-=1
    if j>0: row[j-1]+=1
    A.append(np.sqrt(omega)*row); b.append(np.sqrt(omega)*z); labels.append(label)
A=np.vstack(A); b=np.array(b)
x_est=np.r_[0, np.linalg.lstsq(A,b,rcond=None)[0]]
x_odom=np.r_[0, np.cumsum(odom)]

fig, ax=plt.subplots(figsize=(9,4))
ax.plot(true,np.zeros(N),'ko-',lw=2,label='true')
ax.plot(x_odom,np.ones(N)*0.2,'o-',color='#E85D24',lw=2,label='odometry only')
ax.plot(x_est,np.ones(N)*0.4,'o-',color='#534AB7',lw=2,label='GraphSLAM estimate')
for i,j,z,omega,label in edges:
    y=0.6 if label=='loop' else 0.5
    ax.plot([x_est[i],x_est[j]],[y,y],color='#1D9E75' if label=='loop' else 'gray',alpha=0.7)
ax.set_yticks([]); ax.grid(axis='x',alpha=0.25); ax.legend(); ax.set_title('1D pose graph: loop closure가 drift를 보정')
plt.savefig('assets/07_graph_slam_1d.png',dpi=150,bbox_inches='tight'); plt.show()
print('odom final error:', round(x_odom[-1]-true[-1],4))
print('graph final error:', round(x_est[-1]-true[-1],4))

## 2. Information Matrix 구조

GraphSLAM의 선형화된 normal equation은 다음 형태다.

$$H\Delta x = -b, \qquad H=J^T\Omega J$$

제약이 인접 pose 사이에만 걸리면 $H$는 sparse band matrix가 된다.

In [ ]:
H=A.T@A
fig, ax=plt.subplots(figsize=(5,5))
ax.spy(H, markersize=12, color='#534AB7')
ax.set_title('sparse information / Hessian structure')
plt.savefig('assets/07_information_matrix_sparsity.png',dpi=150,bbox_inches='tight')
plt.show()
print(np.round(H,2))

## 요약

| 개념 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| Pose graph | 상태를 노드, 제약을 edge로 표현 | Ch.10 SLAM, Ch.11 GraphSLAM |
| Information matrix | constraint confidence | Weighted least squares |
| Loop closure | 누적 odometry drift 보정 | SLAM의 핵심 사건 |
| Sparsity | 국소 제약으로 생기는 희소 구조 | 대규모 SLAM 최적화 |